## Phase 4: Transformer-based Text Classification (BERT)
In this phase, a transformer-based language model is applied to the support ticket classification task. Instead of learning word representations from scratch, a pretrained BERT model is fine-tuned on the dataset.

The model leverages contextualized word embeddings and self-attention mechanisms to capture semantic relationships across the entire input sequence. This approach is expected to improve classification performance, especially under limited data conditions.

#### Load and preprocess the cleaned data
We use the cleaned dataset from Phase 1. The text data is stored in the 'clean_text' column and the target labels (departments) are in the 'queue' column.

In [27]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("../data/dataset_en_clean.csv")
texts = df["clean_text"].values

label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(df["label"].values)

#### Transformer Tokenization (DistilBERT)
For the transformer-based model, a pretrained tokenizer from the DistilBERT model is used. Unlike the custom tokenizer employed in previous phases, this tokenizer is trained on large-scale corpora and applies subword tokenization (WordPiece), allowing it to handle unknown and rare words effectively.

The tokenizer converts raw text into numerical input representations consisting of two main components: input IDs and attention masks. The input IDs correspond to token indices in the pretrained vocabulary, while the attention mask distinguishes between real tokens and padding tokens. Padding and truncation are applied to ensure a fixed sequence length for all inputs.

In [28]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [29]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

#### Dataset and DataLoader

Similar to phase3, we once again need to implement a ``Dataset`` and ``Dataloader``. To interface with the transformer model, a custom PyTorch Dataset is implemented. For each text sample, the pretrained tokenizer is applied to generate input IDs and attention masks with fixed sequence length. The corresponding class label is converted into a tensor.

In [30]:
import torch
from torch.utils.data import Dataset, DataLoader

class TicketDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }

The DataLoader wraps the dataset and provides mini-batches during training, enabling efficient optimization and shuffling of the data.

In [31]:
train_dataset = TicketDataset(
    train_texts,
    train_labels,
    tokenizer,
    max_length=128
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

#### Transformer-based Classification Model

A pretrained DistilBERT model is used as the backbone of the classification architecture. The model encodes the input text into contextualized token representations using self-attention mechanisms. The representation of the special [CLS] token is used as a fixed-size summary of the input sequence and is passed to a linear classification layer to predict the target class.

The model is fine-tuned end-to-end on the labeled dataset, allowing both the pretrained language representations and the classification head to adapt to the specific task.


In [32]:
from transformers import DistilBertForSequenceClassification

num_classes = 10

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_classes
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


#### Model Training (Transformer)

The transformer-based model is fine-tuned using the AdamW optimizer with a small learning rate to prevent catastrophic forgetting of pretrained representations. During training, the model receives tokenized inputs along with attention masks and class labels. The loss is computed internally by the model using a cross-entropy objective.

Gradients are backpropagated through the entire network, allowing both the classification head and the pretrained transformer layers to adapt to the specific classification task.

In [33]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
epochs = 15

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{epochs}, Training Loss: {avg_loss:.4f}")


Epoch 1/15: 100%|██████████| 817/817 [00:46<00:00, 17.76it/s]


Epoch 1/15, Training Loss: 0.8488


Epoch 2/15: 100%|██████████| 817/817 [00:46<00:00, 17.56it/s]


Epoch 2/15, Training Loss: 0.7064


Epoch 3/15: 100%|██████████| 817/817 [00:46<00:00, 17.60it/s]


...

Epoch 13/15, Training Loss: 0.0488


Epoch 14/15: 100%|██████████| 817/817 [00:46<00:00, 17.60it/s]


Epoch 14/15, Training Loss: 0.0403


Epoch 15/15: 100%|██████████| 817/817 [00:44<00:00, 18.18it/s]

Epoch 15/15, Training Loss: 0.0432


#### Model Evaluation (Transformer)

After training, the transformer-based model is evaluated on the validation set. The model is switched to evaluation mode to disable dropout layers, and gradient computation is disabled to improve efficiency.

Predicted class labels are obtained by selecting the class with the highest output logit for each input sample. Standard classification metrics, including precision, recall, F1-score, and the confusion matrix, are computed to assess model performance and analyze class-wise behavior.

In [34]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import torch

val_dataset = TicketDataset(
    val_texts,
    val_labels,
    tokenizer,
    max_length=128
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print(classification_report(all_labels, all_preds))
print(confusion_matrix(all_labels, all_preds))


              precision    recall  f1-score   support

           0       0.81      0.84      0.83      1580
           1       0.83      0.83      0.83      1469
           2       0.78      0.59      0.67       219

    accuracy                           0.82      3268
   macro avg       0.81      0.75      0.78      3268
weighted avg       0.82      0.82      0.82      3268

[[1334  220   26]
 [ 245 1214   10]
 [  59   31  129]]


In [35]:
path = "../models/bert_classifier"

model.save_pretrained(path)
tokenizer.save_pretrained(path)

('../models/bert_classifier/tokenizer_config.json',
 '../models/bert_classifier/special_tokens_map.json',
 '../models/bert_classifier/vocab.txt',
 '../models/bert_classifier/added_tokens.json',
 '../models/bert_classifier/tokenizer.json')